In [1]:
!pip install bert-score sentence-transformers torch numpy

In [2]:
from bert_score import score
from sentence_transformers import SentenceTransformer, util
import numpy as np

In [3]:
question = "Who invented the telephone?"

reference_answer = """
Alexander Graham Bell is credited with inventing the first practical telephone.
He received the patent in 1876.
"""

llm_answer = """
Alexander Graham Bell invented the telephone in 1876.
"""

In [4]:
model = SentenceTransformer("all-MiniLM-L6-v2")

In [5]:
#BERTScore
P, R, F1 = score(
    [llm_answer],
    [reference_answer],
    lang="en"
)

bert_score = F1.mean().item()

print("BERTScore:", bert_score)

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/482 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/1.42G [00:00<?, ?B/s]

Some weights of RobertaModel were not initialized from the model checkpoint at roberta-large and are newly initialized: ['roberta.pooler.dense.bias', 'roberta.pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


BERTScore: 0.9424732327461243


In [6]:
#FactScore

answer_embedding = model.encode(
    llm_answer,
    convert_to_tensor=True
)

reference_embedding = model.encode(
    reference_answer,
    convert_to_tensor=True
)

fact_score = util.cos_sim(
    answer_embedding,
    reference_embedding
).item()

print("FactScore:", fact_score)

FactScore: 0.8882997035980225


In [7]:
#SelfCheckGPT

multiple_answers = [
    "Alexander Graham Bell invented the telephone in 1876.",
    "The telephone was invented by Alexander Graham Bell.",
    "Bell received a patent for the telephone in 1876.",
    "Alexander Graham Bell created the first practical telephone."
]

embeddings = model.encode(
    multiple_answers,
    convert_to_tensor=True
)

scores = []

for i in range(len(embeddings)):
    for j in range(i + 1, len(embeddings)):
        similarity = util.cos_sim(
            embeddings[i],
            embeddings[j]
        ).item()

        scores.append(similarity)

selfcheck_score = np.mean(scores)

print("SelfCheckGPT:", selfcheck_score)

SelfCheckGPT: 0.85018918911616


In [8]:
#Final Verdict

if (
    bert_score > 0.85 and
    fact_score > 0.75 and
    selfcheck_score > 0.80
):
    verdict = "NO HALLUCINATION"
else:
    verdict = "POTENTIAL HALLUCINATION"

print("Verdict:", verdict)

Verdict: NO HALLUCINATION


In [9]:
#Hallucinated Example

hallucinated_answer = """
Albert Einstein invented the telephone in 1910.
"""

P, R, F1 = score(
    [hallucinated_answer],
    [reference_answer],
    lang="en"
)

print("Hallucinated BERTScore:", F1.mean().item())

Some weights of RobertaModel were not initialized from the model checkpoint at roberta-large and are newly initialized: ['roberta.pooler.dense.bias', 'roberta.pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Hallucinated BERTScore: 0.8997519016265869
